# Combined MINLPLib + QPLIB — 64 Static DI Features

Run **Cell 1**, then **Cell 2**. Paths in Cell 2 are the same as the existing corrected `.py` scripts; change them only if your files are elsewhere. The 64-feature calculations are unchanged.

In [4]:
class QCQPInstance:
    pass
import pickle

with open('instances_mixed8.pkl', 'rb') as f:
    instances = pickle.load(f)

# Convert to dicts
instances = [inst.__dict__ for inst in instances]




In [5]:
import hashlib
"""
============================================================
MINLPLIB FEATURE EXTRACTION - DI + EPS-SAFE + SPECTRUM-SCALED
(Pruned extras: keep informative, drop saturated/redundant)
============================================================
- Extra feature sets can be toggled on/off
- Handles multiple PKL directories
- Handles multiple objective files
- Automatic alignment
- EPS-safe ratios + spectrum-scaled eigenvalue features
============================================================

Pruning policy (default):
  ✅ Keep:  comm_mean, comm_max
  ✅ Keep:  LogCond aggregated stats: avg, IQR, skewness, CV
  ❌ Drop:  comm_min
  ❌ Drop:  diag_off_ratio extras by default (often redundant/saturated)
  ❌ Drop:  log10_cond_A0 by default (often saturated when A0 is zero)
  ✅ Add:   g_overlap_Theta, g_indef_Theta, g_linear_mass (robust partition)

CHANGES vs previous version:
  1) Raw lambda_min_A0 / lambda_max_A0 removed; scaled versions kept
  2) Raw Lambda_min / Lambda_max removed from constraint aggregations;
     Lambda_min_scaled / Lambda_max_scaled kept
  3) mu3 (raw 3rd central moment) replaced by
     skewness = mu3 / (std^3 + EPS_SKEW)  — dimensionless, scale-invariant
  4) CV = std / (|mean| + EPS_CV)  — always non-negative denominator
============================================================
"""

import random
from time import perf_counter
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
from pathlib import Path
from typing import List, Tuple


# ============================================================
# GLOBAL SWITCHES
# ============================================================

INCLUDE_NM_COLUMNS = False
THETA_CLASS_EPS    = 1e-8

DEFAULT_EXTRA_FEATURES = ("cond", "commutator")
INCLUDE_COND_A0        = False
INCLUDE_DIAG_OFF_RATIO = False
INCLUDE_SCALE_FEATURES = False

KEEP_LOGCOND_STATS    = ("avg", "IQR", "skewness", "CV")
KEEP_LOGDIAGOFF_STATS = ("avg", "IQR", "skewness", "CV")

ADD_THETA_PARTITION_FEATURES = True
MASS_TOL = 1e-10


# ============================================================
# EPS / TINY CONSTANTS
# ============================================================

EPS_NUM   = 1e-6
EPS_THETA = 1e-6
EPS_CV    = 1e-6      # stabilizer for CV denominator  |mean| + EPS_CV
EPS_SKEW  = 1e-9      # stabilizer for skewness denominator  std^3 + EPS_SKEW
EPS_SPEC  = 1e-6
EPS_EIG   = 1e-12


# ============================================================
# OBJECTIVE FILE HANDLING
# ============================================================

def _read_table(path: str) -> pd.DataFrame:
    path = str(path)
    suf = Path(path).suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file: {path}")


def _write_table(df: pd.DataFrame, path: str):
    path = str(path)
    suf = Path(path).suffix.lower()
    if suf == ".csv":
        df.to_csv(path, index=False)
        return
    if suf in (".xlsx", ".xls"):
        df.to_excel(path, index=False)
        return
    raise ValueError(f"Unsupported output: {path}")


def _canon_key(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = str(x).strip()
    s = s.replace(".pkl", "")
    return s


def _ensure_objective_id_global(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "id_global" in df.columns:
        base = df["id_global"].map(_canon_key)
        has_double = base.astype(str).str.contains("__", regex=False)

        if ("id" in df.columns) and (not has_double.any()):
            df["id_global"] = base + "__" + df["id"].astype(str)
        else:
            if "id" in df.columns:
                dup = base.duplicated(keep=False)
                if dup.any():
                    df.loc[dup, "id_global"] = (
                        base.loc[dup] + "__" + df.loc[dup, "id"].astype(str)
                    )
                else:
                    df["id_global"] = base
            else:
                df["id_global"] = base
        return df

    if ("source_pkl" in df.columns) and ("id" in df.columns):
        stems = df["source_pkl"].astype(str).map(
            lambda s: Path(s).stem.replace(".pkl", "")
        )
        df["id_global"] = stems + "__" + df["id"].astype(str)
        return df

    if ("instance_name" in df.columns) and ("id" in df.columns):
        df["id_global"] = (
            df["instance_name"].astype(str).map(_canon_key)
            + "__"
            + df["id"].astype(str)
        )
        return df

    raise ValueError("Cannot create id_global")


def _coerce_obj_col(df: pd.DataFrame, col: str):
    if col not in df.columns:
        return
    df[col] = df[col].replace(
        {"#NAME?": np.nan, "inf": np.inf, "-inf": -np.inf, "nan": np.nan}
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")


def _fix_unbounded(
    df: pd.DataFrame, prefixes=("lp", "sdp", "sdpp", "sdp_rlt")
) -> pd.DataFrame:
    df = df.copy()
    for p in prefixes:
        obj = f"{p}_obj"
        st  = f"{p}_status"
        if obj in df.columns:
            _coerce_obj_col(df, obj)
        if (obj in df.columns) and (st in df.columns):
            s = df[st].astype(str).str.lower()
            mask = s.str.contains("unbounded") | s.str.contains("dual_infeasible")
            df.loc[mask, obj] = -np.inf
    return df


def _coalesce_group(g: pd.DataFrame) -> pd.Series:
    def _rank_row(r):
        rank = 0
        for p in ["lp", "sdp", "sdpp", "sdp_rlt"]:
            st = str(r.get(f"{p}_status", ""))
            if st and (
                st.lower() not in ("", "not_run", "skipped")
            ) and (not st.lower().startswith("err_")):
                rank += 1
        return rank

    g = g.copy()
    g["_rank"] = g.apply(_rank_row, axis=1)
    g = g.sort_values("_rank", ascending=False)

    out = {}
    cols = [c for c in g.columns if c != "_rank"]

    for c in cols:
        s = g[c]
        if c.endswith("_status"):
            cand = s.dropna().astype(str)
            cand = cand[
                ~cand.str.lower().isin(["not_run", "skipped", "nan", "none"])
            ]
            out[c] = (
                cand.iloc[0]
                if len(cand)
                else (s.dropna().iloc[0] if s.notna().any() else "not_run")
            )
        else:
            cand = s.dropna()
            out[c] = cand.iloc[0] if len(cand) else np.nan

    return pd.Series(out)


def combine_objective_files(
    objective_paths: List[str], out_path: str
) -> pd.DataFrame:
    dfs = []
    for p in objective_paths:
        d = _read_table(p)
        d = _ensure_objective_id_global(d)
        d["_source_file"] = Path(p).name
        dfs.append(d)
        print(f"  📂 Loaded {Path(p).name}: {len(d)} rows")

    df = pd.concat(dfs, ignore_index=True, sort=False)
    print(f"  🔗 Concatenated: {len(df)} total rows")

    df = _fix_unbounded(df)

    n_before = len(df)
    df = (
        df.groupby("id_global", as_index=False)
        .apply(lambda gg: _coalesce_group(gg))
        .reset_index(drop=True)
    )
    n_after = len(df)

    if n_before != n_after:
        print(f"  ⚠️  Removed {n_before - n_after} duplicate rows (kept best)")

    df = df.sort_values("id_global").reset_index(drop=True)
    _write_table(df, out_path)

    print(f"✅ Combined objectives -> {out_path} | shape={df.shape}")
    return df


# ============================================================
# FEATURE EXTRACTION HELPERS
# ============================================================

def _sym(A):
    A = np.asarray(A, dtype=float)
    return 0.5 * (A + A.T)


def detect_sparsity_patterns(A, tol=1e-8):
    patterns = {}
    A = _sym(A)
    n = A.shape[0]
    off_diag = A - np.diag(np.diag(A))

    patterns["Diagonal"] = bool(np.allclose(off_diag, 0, atol=tol))
    patterns["Hollow"]   = bool(np.allclose(np.diag(A), 0, atol=tol))

    patterns["Bipartite"] = False
    patterns["Tree"]      = False
    patterns["Chordal"]   = False
    patterns["Planar"]    = False

    mask = (np.abs(np.triu(off_diag, 1)) > tol)
    if np.any(mask):
        rows, cols = np.where(mask)
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edges_from(zip(rows.tolist(), cols.tolist()))

        patterns["Bipartite"] = bool(bipartite.is_bipartite(G))
        patterns["Tree"]      = bool(nx.is_tree(G))
        planar, _             = nx.check_planarity(G)
        patterns["Planar"]    = bool(planar)
        patterns["Chordal"]   = bool(nx.is_chordal(G))

    return patterns


def compute_statistics(vector):
    """
    CHANGE 1: mu3 replaced by skewness = mu3 / (std^3 + EPS_SKEW)
              Dimensionless and scale-invariant; mu3 has units (value)^3
              making it incomparable across differently-scaled instances.

    CHANGE 2: CV = std / (|mean| + EPS_CV)
              Original std/mean is signed when mean < 0 (common for
              indefinite constraint eigenvalue distributions), producing
              meaningless negative "relative variability" values.
    """
    if vector is None or len(vector) == 0:
        return {
            "min": 0.0, "max": 0.0, "avg": 0.0,
            "skewness": 0.0, "IQR": 0.0, "Pout": 0.0, "CV": 0.0,
        }

    v = np.asarray(vector, dtype=float).reshape(-1)
    mean_val = float(np.mean(v))
    std_val  = float(np.std(v))

    # --- skewness (dimensionless, scale-invariant) ---
    if v.size > 1:
        mu3 = float(np.mean((v - mean_val) ** 3))
        skewness = float(mu3 / (std_val ** 3 + EPS_SKEW))
    else:
        skewness = 0.0

    q1, q3 = np.percentile(v, [25, 75])
    iqr = float(q3 - q1)

    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        pout = float(np.mean((v < lower) | (v > upper)))
    else:
        pout = 0.0

    # --- CV with non-negative denominator ---
    cv = float(std_val / (abs(mean_val) + EPS_CV))

    return {
        "min":      float(np.min(v)),
        "max":      float(np.max(v)),
        "avg":      float(mean_val),
        "skewness": skewness,
        "IQR":      float(iqr),
        "Pout":     float(pout),
        "CV":       cv,
    }


def _as_dict_instance(inst):
    if isinstance(inst, dict):
        return inst
    if hasattr(inst, "__dict__"):
        return inst.__dict__
    raise TypeError("Instance is neither dict nor object with __dict__")


def _load_pkl_payload(pkl_path: str):
    obj = pd.read_pickle(pkl_path)
    return obj if isinstance(obj, list) else [obj]


# ============================================================
# EXTRA FEATURE FUNCTIONS
# ============================================================

def _log10_cond_from_eigs(eigs, eig_eps=EPS_EIG):
    a = np.abs(np.asarray(eigs, dtype=float))
    amax = float(np.max(a)) if a.size else 1.0
    nz = a[a > eig_eps]
    if nz.size == 0:
        cond = 1e16
    else:
        amin = float(np.min(nz))
        cond = amax / (amin + EPS_NUM)
    cond = max(1.0, min(cond, 1e16))
    return float(np.log10(cond))


def _log10_diag_off_energy_ratio(A, tiny=EPS_NUM):
    A = _sym(A)
    d = np.diag(A)
    diag_e = float(np.sum(d * d))
    tot_e  = float(np.sum(A * A))
    off_e  = max(tot_e - diag_e, 0.0)

    if off_e < tiny:
        ratio = 1e16
    elif diag_e < tiny:
        ratio = 1e-6
    else:
        ratio = diag_e / (off_e + tiny)

    ratio = max(1e-6, min(ratio, 1e16))
    return float(np.log10(ratio))


def _commutator_norm(A, B, tiny=EPS_NUM):
    A = _sym(A)
    B = _sym(B)
    C = A @ B - B @ A
    num = float(np.linalg.norm(C, ord="fro"))
    den = float(np.linalg.norm(A, ord="fro") * np.linalg.norm(B, ord="fro"))
    return num / (den + tiny)


def _compute_commutator_features(A_list, max_pairs=40, seed_key=""):
    # Pruned: keep only mean/max (drop comm_min)
    if len(A_list) < 2:
        return {"comm_mean": 0.0, "comm_max": 0.0}

    mats = [_sym(A) for A in A_list]
    norms = []

    for k in range(1, len(mats)):
        norms.append(_commutator_norm(mats[0], mats[k], tiny=EPS_NUM))

    if len(mats) > 2:
        total_pairs = (len(mats) - 1) * (len(mats) - 2) // 2
        if total_pairs <= max_pairs:
            for i in range(1, len(mats)):
                for j in range(i + 1, len(mats)):
                    norms.append(_commutator_norm(mats[i], mats[j], tiny=EPS_NUM))
        else:
            all_pairs = [
                (i, j)
                for i in range(1, len(mats))
                for j in range(i + 1, len(mats))
            ]
            # Local, stable seed; no dependence on traversal/order or global RNG.
            stable = int.from_bytes(hashlib.sha256(str(seed_key).encode("utf-8")).digest()[:8], "little")
            sampled = random.Random(stable).sample(all_pairs, min(max_pairs, len(all_pairs)))
            for i, j in sampled:
                norms.append(_commutator_norm(mats[i], mats[j], tiny=EPS_NUM))

    if len(norms) == 0:
        norms = [0.0]

    norms = np.asarray(norms, dtype=float)
    return {"comm_mean": float(np.mean(norms)), "comm_max": float(np.max(norms))}


def _sample_abs_entries(arr, cap=200000, tiny=EPS_NUM):
    v = np.abs(np.asarray(arr, dtype=float).ravel())
    v = v[np.isfinite(v)]
    v = v[v > tiny]
    if v.size == 0:
        return v
    if v.size <= cap:
        return v
    step = int(np.ceil(v.size / cap))
    return v[::step]


def _scale_features_from_values(v, tiny=EPS_NUM):
    if v.size == 0:
        return {
            "scale_max_abs": 0.0,
            "scale_min_pos_abs": 0.0,
            "scale_median_abs": 0.0,
            "scale_log10_range": 0.0,
            "scale_max_over_median": 0.0,
            "scale_p95_over_p05": 0.0,
        }

    vmax = float(np.max(v))
    vmin = float(np.min(v))
    vmed = float(np.median(v))
    p05  = float(np.percentile(v, 5))  if v.size >= 20 else vmin
    p95  = float(np.percentile(v, 95)) if v.size >= 20 else vmax

    return {
        "scale_max_abs":         vmax,
        "scale_min_pos_abs":     vmin,
        "scale_median_abs":      vmed,
        "scale_log10_range":     float(np.log10((vmax + tiny) / (vmin + tiny))),
        "scale_max_over_median": float((vmax + tiny) / (vmed + tiny)),
        "scale_p95_over_p05":    float((p95 + tiny) / (p05 + tiny)),
    }


def _bounds_exist_from_pkl(inst: dict, source_label: str = "") -> int:
    """Original-variable bound status, never tightened/working/artificial bounds.

    QPLIB requires an all-variable raw bound check, not the quadratic-only flag.
    MINLPLib retains the prior flag precedence to preserve its definition.
    """
    if not isinstance(inst, dict):
        inst = dict(vars(inst))
    source = str(source_label).strip().upper()

    def all_raw_bounds():
        if inst.get("all_vars_raw_bounded") is not None:
            return int(bool(inst["all_vars_raw_bounded"]))
        if inst.get("l_raw") is not None and inst.get("u_raw") is not None:
            lo = np.asarray(inst["l_raw"], dtype=float).reshape(-1)
            up = np.asarray(inst["u_raw"], dtype=float).reshape(-1)
            if lo.size and lo.shape == up.shape:
                return int(bool(np.isfinite(lo).all() and np.isfinite(up).all()))
        info = inst.get("bounds_tightening_info")
        if isinstance(info, dict) and (
            "missing_lb_raw_idx" in info or "missing_ub_raw_idx" in info
        ):
            return int(not (
                len(info.get("missing_lb_raw_idx", []))
                + len(info.get("missing_ub_raw_idx", []))
            ))
        return None

    if source == "QPLIB":
        found = all_raw_bounds()
        if found is None:
            raise ValueError(
                "QPLIB raw all-variable bounds metadata unavailable; "
                "do not substitute quadratic-only or tightened bounds."
            )
        return found

    # Historical MINLPLib precedence retained.
    if inst.get("bounds_exist") is not None:
        return int(bool(inst["bounds_exist"]))
    if inst.get("bounds_exist_raw_for_quad") is not None:
        return int(bool(inst["bounds_exist_raw_for_quad"]))
    found = all_raw_bounds()
    if found is not None:
        return found
    raise ValueError("Original-bound metadata missing: cannot infer BoundsExist")


def _theta_and_masses_from_eigs(eigs, tol, eps_denom=EPS_THETA):
    eigs = np.asarray(eigs, dtype=float)
    a = np.abs(eigs)
    total_mass = float(np.sum(a))
    neg_mass   = float(np.sum(a[eigs < -tol]))
    pos_mass   = float(np.sum(a[eigs >  tol]))
    theta      = float(neg_mass / (total_mass + eps_denom))
    return theta, total_mass, neg_mass, pos_mass


def _spec_scale_denom(eigs, eps=EPS_SPEC):
    """sum(|eigenvalues|) + eps — used to normalise individual eigenvalues."""
    return float(np.sum(np.abs(np.asarray(eigs, dtype=float))) + eps)


# ============================================================
# MAIN FEATURE EXTRACTION (MINLPLIB PKL FOLDERS)
# ============================================================

def extract_features_DI_for_folder(
    pkl_dir: str,
    source_label: str,
    tol=1e-8,
    extra_feature_sets=None,
    comm_max_pairs=40,
    scale_cap=200000,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    Extract the SAME 64 DI features as before.

    Only additions relative to the original code:
      1) source_label is used only to create collision-safe IDs.
      2) feature extraction time is measured per instance and written separately.

    IMPORTANT:
      - source_label is NOT added to the ML feature file.
      - timing columns are NOT added to the ML feature file.
      - feature definitions are unchanged.
    """

    if extra_feature_sets is None:
        extra_feature_sets = DEFAULT_EXTRA_FEATURES

    extra_feature_sets = set(extra_feature_sets or [])
    if not INCLUDE_DIAG_OFF_RATIO:
        extra_feature_sets.discard("diag_off_ratio")
    if not INCLUDE_SCALE_FEATURES:
        extra_feature_sets.discard("scale")

    pkl_dir = Path(pkl_dir)
    pkl_files = sorted(pkl_dir.glob("*.pkl"))
    if not pkl_files:
        raise FileNotFoundError(f"No .pkl files found in {pkl_dir}")

    rows = []
    timing_rows = []

    sparsity_patterns = ["Diagonal", "Hollow", "Bipartite", "Tree", "Chordal", "Planar"]
    stat_names = ["min", "max", "avg", "skewness", "IQR", "Pout", "CV"]

    print(
        f"  📂 Processing {len(pkl_files)} PKL files "
        f"from {pkl_dir.name} [{source_label}]..."
    )

    for p in pkl_files:
        src_stem = p.stem

        # Keep PKL loading outside the feature-computation timer.
        # This lets us report pure feature computation cost separately from I/O.
        t_load0 = perf_counter()
        payload = _load_pkl_payload(str(p))
        file_load_time = perf_counter() - t_load0

        for local_idx, inst in enumerate(payload):
            inst = _as_dict_instance(inst)
            rid = inst.get("instance_id", inst.get("id", local_idx))

            # Collision-safe common ID across MINLPLib and QPLIB.
            id_global = f"{source_label}__{src_stem}__{rid}"

            A_list = inst["A_list"]
            b_list = inst.get("b_list", None)
            c_list = inst.get("c_list", None)

            n = int(len(A_list[0]))
            m = int(len(A_list) - 1)

            bounds_exist = _bounds_exist_from_pkl(inst, source_label=source_label)

            # -------------------------------------------------
            # Reviewer timing: PURE feature extraction starts
            # -------------------------------------------------
            t_feat0 = perf_counter()

            row = {
                "id": id_global,
                "instance_name": src_stem,
                "BoundsExist": int(bounds_exist),
            }
            if INCLUDE_NM_COLUMNS:
                row["n"] = n
                row["m"] = m

            # ----------------------------------------
            # A0 features
            # ----------------------------------------
            A0 = _sym(A_list[0])
            eigs0 = np.linalg.eigvalsh(A0)

            neg0 = np.sum(eigs0 < -tol)
            row["eta_A0_over_n"] = float(neg0 / n)

            theta0, total_mass0, neg_mass0, pos_mass0 = _theta_and_masses_from_eigs(
                eigs0, tol=tol, eps_denom=EPS_THETA
            )
            row["theta_A0"] = float(theta0)

            # SCALED extremal eigenvalues only
            den_spec0 = _spec_scale_denom(eigs0, eps=EPS_SPEC)
            row["lambda_min_A0_scaled"] = float(np.min(eigs0) / den_spec0)
            row["lambda_max_A0_scaled"] = float(np.max(eigs0) / den_spec0)

            row["rho_A0_over_n"] = float(np.linalg.matrix_rank(A0) / n)

            patt0 = detect_sparsity_patterns(A0, tol=tol)
            for pat in sparsity_patterns:
                row[f"P{pat[0]}_A0"] = int(patt0[pat])

            if ("cond" in extra_feature_sets) and INCLUDE_COND_A0:
                row["log10_cond_A0"] = _log10_cond_from_eigs(
                    eigs0, eig_eps=EPS_EIG
                )

            if "diag_off_ratio" in extra_feature_sets:
                row["log10_diagoff_A0"] = _log10_diag_off_energy_ratio(
                    A0, tiny=EPS_NUM
                )

            # ----------------------------------------
            # Constraints aggregation
            # ----------------------------------------
            if m > 0:
                Xi, Theta = [], []
                Lambda_min_scaled, Lambda_max_scaled = [], []
                Rho = []
                LogConds, LogDiagOffs = [], []
                sparsity_vectors = {pat: [] for pat in sparsity_patterns}
                scale_vals = []

                convex_flags, concave_flags = [], []
                overlap_flags, indef_flags, linear_flags = [], [], []

                if "scale" in extra_feature_sets:
                    scale_vals.append(
                        _sample_abs_entries(A0, cap=scale_cap, tiny=EPS_NUM)
                    )

                for k in range(1, len(A_list)):
                    A = _sym(A_list[k])
                    eigs = np.linalg.eigvalsh(A)

                    neg = np.sum(eigs < -tol)
                    Xi.append(float(neg / n))

                    th, total_mass, neg_mass, pos_mass = _theta_and_masses_from_eigs(
                        eigs, tol=tol, eps_denom=EPS_THETA
                    )
                    Theta.append(float(th))

                    is_linear = (total_mass <= MASS_TOL)
                    is_convex = (
                        neg_mass
                        <= THETA_CLASS_EPS * (total_mass + EPS_THETA)
                    )
                    is_concave = (
                        pos_mass
                        <= THETA_CLASS_EPS * (total_mass + EPS_THETA)
                    )

                    linear_flags.append(bool(is_linear))
                    convex_flags.append(bool(is_convex))
                    concave_flags.append(bool(is_concave))
                    overlap_flags.append(bool(is_convex and is_concave))
                    indef_flags.append(bool((not is_convex) and (not is_concave)))

                    den_spec = _spec_scale_denom(eigs, eps=EPS_SPEC)
                    Lambda_min_scaled.append(float(np.min(eigs) / den_spec))
                    Lambda_max_scaled.append(float(np.max(eigs) / den_spec))

                    Rho.append(float(np.linalg.matrix_rank(A) / n))

                    if "cond" in extra_feature_sets:
                        LogConds.append(
                            _log10_cond_from_eigs(eigs, eig_eps=EPS_EIG)
                        )

                    if "diag_off_ratio" in extra_feature_sets:
                        LogDiagOffs.append(
                            _log10_diag_off_energy_ratio(A, tiny=EPS_NUM)
                        )

                    if "scale" in extra_feature_sets:
                        scale_vals.append(
                            _sample_abs_entries(A, cap=scale_cap, tiny=EPS_NUM)
                        )

                    patt = detect_sparsity_patterns(A, tol=tol)
                    for pat in sparsity_patterns:
                        sparsity_vectors[pat].append(int(patt[pat]))

                row["g0_Theta"] = (
                    float(np.mean(convex_flags)) if convex_flags else 0.0
                )
                row["g1_Theta"] = (
                    float(np.mean(concave_flags)) if concave_flags else 0.0
                )

                if ADD_THETA_PARTITION_FEATURES:
                    row["g_overlap_Theta"] = (
                        float(np.mean(overlap_flags)) if overlap_flags else 0.0
                    )
                    row["g_indef_Theta"] = (
                        float(np.mean(indef_flags)) if indef_flags else 0.0
                    )
                    row["g_linear_mass"] = (
                        float(np.mean(linear_flags)) if linear_flags else 0.0
                    )

                properties = {
                    "Xi": Xi,
                    "Theta": Theta,
                    "Lambda_min_scaled": Lambda_min_scaled,
                    "Lambda_max_scaled": Lambda_max_scaled,
                    "Rho": Rho,
                }

                for prop_name, vec in properties.items():
                    stats = compute_statistics(vec)
                    for s in stat_names:
                        row[f"h_{s}_{prop_name}"] = stats[s]

                if "cond" in extra_feature_sets:
                    cstats = compute_statistics(LogConds)
                    for s in KEEP_LOGCOND_STATS:
                        row[f"h_{s}_LogCond"] = cstats[s]

                if "diag_off_ratio" in extra_feature_sets:
                    rstats = compute_statistics(LogDiagOffs)
                    for s in KEEP_LOGDIAGOFF_STATS:
                        row[f"h_{s}_LogDiagOff"] = rstats[s]

                for pat in sparsity_patterns:
                    row[f"g{pat[0]}_Phi{pat[0]}"] = float(
                        sum(sparsity_vectors[pat]) / m
                    )

                if "scale" in extra_feature_sets:
                    if b_list is not None:
                        for bk in b_list:
                            scale_vals.append(
                                _sample_abs_entries(
                                    bk, cap=scale_cap, tiny=EPS_NUM
                                )
                            )

                    if c_list is not None:
                        scale_vals.append(
                            _sample_abs_entries(
                                np.asarray(c_list, dtype=float),
                                cap=scale_cap,
                                tiny=EPS_NUM,
                            )
                        )

                    v = (
                        np.concatenate(
                            [x for x in scale_vals if x.size > 0], axis=0
                        )
                        if any(x.size > 0 for x in scale_vals)
                        else np.array([])
                    )
                    row.update(_scale_features_from_values(v, tiny=EPS_NUM))

            else:
                row["g0_Theta"] = 0.0
                row["g1_Theta"] = 0.0

                if ADD_THETA_PARTITION_FEATURES:
                    row["g_overlap_Theta"] = 0.0
                    row["g_indef_Theta"] = 0.0
                    row["g_linear_mass"] = 0.0

                for prop_name in [
                    "Xi",
                    "Theta",
                    "Lambda_min_scaled",
                    "Lambda_max_scaled",
                    "Rho",
                ]:
                    for s in stat_names:
                        row[f"h_{s}_{prop_name}"] = 0.0

                if "cond" in extra_feature_sets:
                    for s in KEEP_LOGCOND_STATS:
                        row[f"h_{s}_LogCond"] = 0.0

                if "diag_off_ratio" in extra_feature_sets:
                    for s in KEEP_LOGDIAGOFF_STATS:
                        row[f"h_{s}_LogDiagOff"] = 0.0

                for pat in sparsity_patterns:
                    row[f"g{pat[0]}_Phi{pat[0]}"] = 0.0

                if "scale" in extra_feature_sets:
                    row.update(
                        _scale_features_from_values(
                            np.array([]), tiny=EPS_NUM
                        )
                    )

            if "commutator" in extra_feature_sets:
                row.update(
                    _compute_commutator_features(
                        A_list, max_pairs=comm_max_pairs,
                        seed_key=id_global
                    )
                )

            feature_time = perf_counter() - t_feat0
            # -------------------------------------------------
            # Reviewer timing: PURE feature extraction ends
            # -------------------------------------------------

            rows.append(row)

            # Timing is kept OUTSIDE the feature table.
            timing_rows.append(
                {
                    "id": id_global,
                    "source": source_label,
                    "instance_name": src_stem,
                    "n": n,
                    "m": m,
                    "feature_extraction_time_sec": float(feature_time),
                    # File load time is reported separately. For a PKL containing
                    # multiple instances, this is the file-level load time and is
                    # repeated for audit purposes; do not sum it across rows.
                    "source_pkl_load_time_sec": float(file_load_time),
                }
            )

    df = pd.DataFrame(rows)
    timing_df = pd.DataFrame(timing_rows)

    print(
        f"  ✅ Extracted {len(df)} instances | "
        f"{df.shape[1] - 2} ML features + id + instance_name"
    )

    return df, timing_df

def extract_features_from_multiple_dirs(
    dataset_dirs: List[Tuple[str, str]],
    out_xlsx: str,
    out_timing_csv: str,
    tol=1e-8,
    extra_feature_sets=None,
    comm_max_pairs=40,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    dataset_dirs example:
        [
            ("MINLPLIB", ".../qcp_lpall"),
            ("MINLPLIB", ".../qcp_lpopf"),
            ("MINLPLIB", ".../qcp_lpbigpkl"),
            ("QPLIB",    ".../pickles_nlt300_mge1_bounded"),
        ]

    All directories are processed with the SAME 64-feature extractor.
    """

    if extra_feature_sets is None:
        extra_feature_sets = DEFAULT_EXTRA_FEATURES

    print(f"\n{'='*60}")
    print(f"COMBINED FEATURE EXTRACTION FROM {len(dataset_dirs)} DIRECTORIES")
    print(f"{'='*60}")

    all_features = []
    all_timings = []

    for i, (source_label, pkl_dir) in enumerate(dataset_dirs, 1):
        print(
            f"\n[{i}/{len(dataset_dirs)}] "
            f"{source_label}: {Path(pkl_dir).name}"
        )

        feat, timing = extract_features_DI_for_folder(
            pkl_dir,
            source_label=source_label,
            tol=tol,
            extra_feature_sets=extra_feature_sets,
            comm_max_pairs=comm_max_pairs,
        )

        all_features.append(feat)
        all_timings.append(timing)

    print("\n🔗 Combining all feature tables...")
    combined = pd.concat(all_features, ignore_index=True)
    timings = pd.concat(all_timings, ignore_index=True)

    dup = combined["id"].duplicated(keep=False)
    if dup.any():
        dups = combined.loc[dup, "id"].tolist()
        raise ValueError(
            "Duplicate canonical IDs found. "
            f"First few: {dups[:10]}"
        )

    combined = combined.sort_values("id").reset_index(drop=True)
    timings = timings.sort_values("id").reset_index(drop=True)

    combined.to_excel(out_xlsx, index=False)
    timings.to_csv(out_timing_csv, index=False)

    n_ml_features = len(
        [c for c in combined.columns if c not in {"id", "instance_name"}]
    )

    print(f"\n✅ Combined features saved: {out_xlsx}")
    print(f"   Total instances: {len(combined)}")
    print(f"   ML feature count: {n_ml_features}")
    print(f"   Total columns: {combined.shape[1]}")
    print(f"✅ Feature timing saved: {out_timing_csv}")

    print("\n⏱️ Feature extraction timing summary:")
    for source, g in timings.groupby("source"):
        print(
            f"   {source}: "
            f"n={len(g)}, "
            f"mean={g['feature_extraction_time_sec'].mean():.6f}s, "
            f"median={g['feature_extraction_time_sec'].median():.6f}s, "
            f"max={g['feature_extraction_time_sec'].max():.6f}s"
        )

    print(
        f"   COMBINED: "
        f"n={len(timings)}, "
        f"mean={timings['feature_extraction_time_sec'].mean():.6f}s, "
        f"median={timings['feature_extraction_time_sec'].median():.6f}s, "
        f"max={timings['feature_extraction_time_sec'].max():.6f}s"
    )

    return combined, timings

def build_aligned_ml_files(
    pkl_dirs: List[str],
    objective_files: List[str],
    out_objectives_csv: str,
    out_features_xlsx: str,
    tol=1e-8,
    extra_feature_sets=None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    if extra_feature_sets is None:
        extra_feature_sets = DEFAULT_EXTRA_FEATURES

    print(f"\n{'='*60}")
    print(f"MINLPLIB PIPELINE - ALIGNED WITH SYNTHETIC")
    print(f"{'='*60}")
    print(
        f"✅ Extra features: {sorted(extra_feature_sets) if extra_feature_sets else 'None'}"
    )

    print(f"\n[STEP 1] FEATURE EXTRACTION")
    feat = extract_features_from_multiple_dirs(
        pkl_dirs,
        out_features_xlsx,
        tol=tol,
        extra_feature_sets=extra_feature_sets,
    )

    print(f"\n{'='*60}")
    print(f"[STEP 2] OBJECTIVE COMBINATION")
    print(f"{'='*60}")
    obj = combine_objective_files(objective_files, out_objectives_csv)

    print(f"\n{'='*60}")
    print(f"[STEP 3] ALIGNMENT")
    print(f"{'='*60}")

    feat_ids = set(feat["id"].astype(str))
    obj_ids  = set(obj["id_global"].astype(str))
    common   = sorted(feat_ids.intersection(obj_ids))

    print(f"\n📊 Alignment Statistics:")
    print(f"   Features:   {len(feat_ids):>6} instances")
    print(f"   Objectives: {len(obj_ids):>6} instances")
    print(f"   Common:     {len(common):>6} instances ✅")

    feat_aligned = (
        feat[feat["id"].astype(str).isin(common)]
        .copy()
        .reset_index(drop=True)
    )
    obj_aligned = (
        obj[obj["id_global"].astype(str).isin(common)]
        .copy()
        .reset_index(drop=True)
    )

    feat_aligned.to_excel(out_features_xlsx, index=False)
    obj_aligned.to_csv(out_objectives_csv, index=False)

    print(f"\n{'='*60}")
    print(f"✅ PIPELINE COMPLETE")
    print(f"{'='*60}")
    print(f"📁 Aligned features:   {out_features_xlsx}")
    print(f"   Shape: {feat_aligned.shape}")
    print(f"📁 Aligned objectives: {out_objectives_csv}")
    print(f"   Shape: {obj_aligned.shape}")
    print(f"🔗 Common instances: {len(common)}")

    return feat_aligned, obj_aligned


# ============================================================
# MAIN - USAGE EXAMPLE
# ============================================================



In [6]:

# ========================================================
# INPUTS
# ========================================================
#
# IMPORTANT:
#   - Put ONLY the MINLPLib instances you want to keep in
#     the listed MINLPLib folders before the final run, OR
#     point these paths to your already-filtered folders.
#   - QPLIB path should point to the final bounded set.
#   - The feature extractor itself does NOT eliminate
#     instances by n or by B&B status.
#
# This keeps dataset filtering separate from feature design.

DATASET_DIRS = [
    (
        "MINLPLIB",
        r"C:\Users\suuser\Desktop\minlplib\qcp_lpall",
    ),
    (
        "MINLPLIB",
        r"C:\Users\suuser\Desktop\minlplib\qcp_lpopf",
    ),
    (
        "MINLPLIB",
        r"C:\Users\suuser\Desktop\minlplib\qcp_lpbigpkl",
    ),
    (
        "QPLIB",
        r"C:\Users\suuser\Desktop\minlplib\Areview\qplib\pickles_nlt300_mge1_bounded",
    ),
]

OUT_FEATURES_XLSX = str(Path(r"C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib") / "combined_minlplib_qplib_features_DI_64_v2_raw.xlsx")
OUT_TIMING_CSV = str(Path(r"C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib") / "combined_minlplib_qplib_feature_times_v2.csv")


Path(OUT_FEATURES_XLSX).parent.mkdir(parents=True, exist_ok=True)

# SAME final 64-feature setup used in thesis/manuscript.
EXTRA_FEATURES = ("cond", "commutator")

features, timings = extract_features_from_multiple_dirs(
    DATASET_DIRS,
    OUT_FEATURES_XLSX,
    OUT_TIMING_CSV,
    tol=1e-8,
    extra_feature_sets=EXTRA_FEATURES,
    comm_max_pairs=40,
)

# Hard audit: this final configuration should produce 64 ML features.
model_feature_cols = [
    c for c in features.columns
    if c not in {"id", "instance_name"}
]

assert len(model_feature_cols) == 64, (
    f"Expected 64 ML features, got {len(model_feature_cols)}. "
    "Stop and inspect feature switches before running experiments."
)

print("\n" + "=" * 60)
print("✅ FINAL AUDIT PASSED")
print("=" * 60)
print(f"Instances: {len(features)}")
print(f"ML features: {len(model_feature_cols)}")
print(f"Feature file: {OUT_FEATURES_XLSX}")
print(f"Timing file:  {OUT_TIMING_CSV}")



COMBINED FEATURE EXTRACTION FROM 4 DIRECTORIES

[1/4] MINLPLIB: qcp_lpall
  📂 Processing 157 PKL files from qcp_lpall [MINLPLIB]...
  ✅ Extracted 157 instances | 64 ML features + id + instance_name

[2/4] MINLPLIB: qcp_lpopf
  📂 Processing 4 PKL files from qcp_lpopf [MINLPLIB]...
  ✅ Extracted 4 instances | 64 ML features + id + instance_name

[3/4] MINLPLIB: qcp_lpbigpkl
  📂 Processing 9 PKL files from qcp_lpbigpkl [MINLPLIB]...
  ✅ Extracted 9 instances | 64 ML features + id + instance_name

[4/4] QPLIB: pickles_nlt300_mge1_bounded
  📂 Processing 143 PKL files from pickles_nlt300_mge1_bounded [QPLIB]...
  ✅ Extracted 143 instances | 64 ML features + id + instance_name

🔗 Combining all feature tables...

✅ Combined features saved: C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\combined_minlplib_qplib_features_DI_64_v2_raw.xlsx
   Total instances: 313
   ML feature count: 64
   Total columns: 66
✅ Feature timing saved: C:\Users\suuser\Desktop\minlplib\Areview\combine

In [7]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
FOLDERS = ["qcp_lpall", "qcp_lpopf", "qcp_lpbigpkl"]

def inspect_instance(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "__dict__"):
        return vars(obj)
    return {}

records = []

for name in FOLDERS:
    matches = [p for p in ROOT.rglob(name) if p.is_dir()]
    print(f"\n{name}: {len(matches)} matching directories")

    for folder in matches:
        files = sorted(folder.glob("*.pkl"))
        print(f"  {folder} ({len(files)} PKLs)")

        for path in files:
            try:
                payload = pd.read_pickle(path)
                instances = payload if isinstance(payload, list) else [payload]

                for i, obj in enumerate(instances):
                    d = inspect_instance(obj)

                    solution_keys = [
                        k for k in d
                        if any(term in str(k).lower() for term in (
                            "lp_obj", "sdp_obj", "sdpp_obj",
                            "objective", "solution", "result"
                        ))
                    ]

                    records.append({
                        "folder": name,
                        "file": path.name,
                        "instance_index": i,
                        "solution_keys": ", ".join(map(str, solution_keys)),
                        "has_solution_keys": bool(solution_keys),
                    })

            except Exception as e:
                records.append({
                    "folder": name,
                    "file": path.name,
                    "instance_index": None,
                    "solution_keys": f"READ ERROR: {e}",
                    "has_solution_keys": False,
                })

df = pd.DataFrame(records)

print("\nSUMMARY")
if df.empty:
    print("No matching pickle instances found.")
else:
    print(df.groupby("folder").agg(
        instances=("file", "size"),
        with_solution_keys=("has_solution_keys", "sum")
    ).to_string())

    print("\nINSTANCES WITH POSSIBLE SAVED OBJECTIVES")
    found = df[df["has_solution_keys"]]
    print(found.head(30).to_string(index=False) if len(found)
          else "No saved objective-like keys found.")



qcp_lpall: 1 matching directories
  C:\Users\suuser\Desktop\minlplib\qcp_lpall (157 PKLs)

qcp_lpopf: 1 matching directories
  C:\Users\suuser\Desktop\minlplib\qcp_lpopf (4 PKLs)

qcp_lpbigpkl: 1 matching directories
  C:\Users\suuser\Desktop\minlplib\qcp_lpbigpkl (9 PKLs)

SUMMARY
              instances  with_solution_keys
folder                                     
qcp_lpall           157                 157
qcp_lpbigpkl          9                   9
qcp_lpopf             4                   4

INSTANCES WITH POSSIBLE SAVED OBJECTIVES
   folder              file  instance_index solution_keys  has_solution_keys
qcp_lpall     bayes2_20.pkl               0       results               True
qcp_lpall     bayes2_30.pkl               0       results               True
qcp_lpall     bayes2_50.pkl               0       results               True
qcp_lpall   camshape100.pkl               0       results               True
qcp_lpall     catmix100.pkl               0       results            

In [8]:

from pathlib import Path
import pandas as pd
from pprint import pprint

ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")

examples = [
    ROOT / "qcp_lpall" / "ex3_1_3.pkl",
    ROOT / "qcp_lpopf" / "pooling_adhya1pq.pkl",
    ROOT / "qcp_lpbigpkl" / "bayes2_20.pkl",
]

for path in examples:
    if not path.exists():
        print(f"\nNot found: {path.name}")
        continue

    obj = pd.read_pickle(path)
    if isinstance(obj, list):
        obj = obj[0]
    if not isinstance(obj, dict):
        obj = vars(obj)

    print(f"\n{'=' * 65}")
    print("INSTANCE:", path.name)
    print("RESULTS TYPE:", type(obj.get("results")).__name__)
    pprint(obj.get("results"), depth=4)



INSTANCE: ex3_1_3.pkl
RESULTS TYPE: dict
{}

Not found: pooling_adhya1pq.pkl

Not found: bayes2_20.pkl
